In [1]:
import pandas as pd


# Q1: Build Personalized Knowledge Base
roll_number = 1024170253

last_two_digits = [roll_number % 100 // 10, roll_number % 10]
categories = ["billing", "account", "general"]

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

# Last two digits are 5 and 2.
# 5 % 3 = 2 -> general
# 2 % 3 = 2 -> general
personalized_entries = [
    {
        "question": "how can i update my profile information",
        "answer": "You can update your profile information from the account settings.",
        "keywords": "profile update information",
        "category": categories[last_two_digits[0] % 3]
    },
    {
        "question": "where can i view general account information",
        "answer": "You can view your general account information from the dashboard.",
        "keywords": "account information dashboard",
        "category": categories[last_two_digits[1] % 3]
    }
]



In [2]:
all_entries = fixed_entries + personalized_entries
df = pd.DataFrame(all_entries)

print("Q1: Final 6-row DataFrame")
print(df)




Q1: Final 6-row DataFrame
                                       question  \
0                        what is the annual fee   
1                         how to reset password   
2                   what are your working hours   
3                         how can i pay the fee   
4       how can i update my profile information   
5  where can i view general account information   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  You can update your profile information from t...   
5  You can view your general account information ...   

                        keywords category  
0          fee cost price charge  billing  
1           password reset login  account  
2         hours timing open time  general  
3            pay payment upi fee  billing  
4  

In [ ]:
# Q2: Generate and Score a Hypothesis
def score_query(query, df):
    query_words = query.lower().split()
    results = []

    for index, row in df.iterrows():
        keywords = row["keywords"].lower().split()
        score = sum(word in keywords for word in query_words)

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    return sorted(results, key=lambda x: x["score"], reverse=True)


print("\nQ2: Query Scoring")
query = input("Enter your query: ")
results = score_query(query, df)



In [ ]:
if results:
    for result in results:
        print(result)
else:
    print("No matching entries found.")




In [6]:
# Q3: Find entries belonging to a category
def same_category(category_name, df):
    return df[df["category"] == category_name]


print("\nQ3: Entries belonging to the personalized entry category")
personalized_category = personalized_entries[0]["category"]
category_result = same_category(personalized_category, df)
print(category_result)





Q3: Entries belonging to the personalized entry category
                                       question  \
2                   what are your working hours   
4       how can i update my profile information   
5  where can i view general account information   

                                              answer  \
2                          We are open 9 AM to 5 PM.   
4  You can update your profile information from t...   
5  You can view your general account information ...   

                        keywords category  
2         hours timing open time  general  
4     profile update information  general  
5  account information dashboard  general  


In [ ]:
# Q4: Add a new keyword and save the updated DataFrame
print("\nQ4: Add New Keyword")

entry_index = 0
new_keyword = input("Enter a new keyword: ").strip()

if new_keyword:
    df.loc[entry_index, "keywords"] += " " + new_keyword

filename = f"{roll_number}_faq_data.csv"
df.to_csv(filename, index=False)

print("\nUpdated DataFrame:")
print(df)
print("\nSaved as:", filename)




In [8]:
# Q5: Count FAQ entries per category using groupby
print("\nQ5: Number of FAQ entries per category")

category_counts = df.groupby("category").size()
print(category_counts)





Q5: Number of FAQ entries per category
category
account    1
billing    2
general    3
dtype: int64


In [10]:
# Q6: Modified scoring function with tie handling
def score_query_with_ties(query, df):
    query_words = query.lower().split()
    results = []

    for index, row in df.iterrows():
        keywords = row["keywords"].lower().split()
        score = sum(word in keywords for word in query_words)

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    if not results:
        print("No matching entries found.")
        return

    max_score = max(result["score"] for result in results)

    best_matches = [
        result for result in results
        if result["score"] == max_score
    ]

    print("Highest confidence score:", max_score)

    if len(best_matches) > 1:
        print("Tie detected! All equally matching entries:")
    else:
        print("No tie. Best matching entry:")

    for result in best_matches:
        print(result)




In [11]:
# Demonstration 1: Tie
print("\nQ6: Query producing a tie")
score_query_with_ties("fee", df)




Q6: Query producing a tie
Highest confidence score: 1
Tie detected! All equally matching entries:
{'index': 0, 'question': 'what is the annual fee', 'answer': 'The annual fee is Rs 500.', 'category': 'billing', 'score': 1}
{'index': 3, 'question': 'how can i pay the fee', 'answer': 'You can pay via UPI, card, or net banking.', 'category': 'billing', 'score': 1}


In [12]:
# Demonstration 2: No tie
print("\nQ6: Query not producing a tie")
score_query_with_ties("password", df)



Q6: Query not producing a tie
Highest confidence score: 1
No tie. Best matching entry:
{'index': 1, 'question': 'how to reset password', 'answer': 'Go to Settings > Reset Password.', 'category': 'account', 'score': 1}
